# Nykaa Domain Support Agent — Task 8: Persisted Conversation Memory

## Objective
Persist conversation history to a JSON file and demonstrate two separate cases:

1. **Continuing conversation:** the agent remembers information from an earlier turn and uses it in a later turn.
2. **Fresh conversation:** a new conversation starts without access to the previous conversation's state.

This notebook uses the Task 7 RAG collection (`nykaa_sentence_chunks`) and the Task 6 order lookup logic. The memory store is a local JSON file, with a separate conversation ID for each conversation.

All response composition is deterministic and works in `MOCK_LLM` mode without API keys.

## 1. Install and import dependencies

In [ ]:
%pip install -U langgraph chromadb sentence-transformers pandas numpy

In [ ]:
from pathlib import Path
from typing import TypedDict, Optional, Literal
import json
import re
from datetime import datetime, timezone

import pandas as pd
import numpy as np
import chromadb
from sentence_transformers import SentenceTransformer
from langgraph.graph import StateGraph, START, END

PROJECT_DIR = Path.cwd()
MEMORY_PATH = PROJECT_DIR / "conversation_memory.json"
CHROMA_DIR = PROJECT_DIR / "chroma_db"
ORDERS_PATH = PROJECT_DIR / "orders_with_escalation_scores.csv"

if not ORDERS_PATH.exists():
    ORDERS_PATH = PROJECT_DIR / "orders.csv"

if not ORDERS_PATH.exists():
    raise FileNotFoundError("Place orders.csv or orders_with_escalation_scores.csv in the project directory.")

if not CHROMA_DIR.exists():
    raise FileNotFoundError("chroma_db not found. Run Task 3 first.")

print("Project directory:", PROJECT_DIR.resolve())
print("Memory JSON:", MEMORY_PATH.resolve())
print("Orders file:", ORDERS_PATH.resolve())


## 2. Connect to the RAG collection and load orders

We use the sentence-based collection selected after Task 5 evaluation.

In [ ]:
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
COLLECTION_NAME = "nykaa_sentence_chunks"
TOP_K = 3

chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))
policy_collection = chroma_client.get_collection(COLLECTION_NAME)
embedding_model = SentenceTransformer(MODEL_NAME)

orders_df = pd.read_csv(ORDERS_PATH)
required = {"record_id", "status", "order_value_inr", "days_since_created", "delayed_shipment"}
missing = required - set(orders_df.columns)
if missing:
    raise ValueError(f"Orders data is missing columns: {sorted(missing)}")

orders_df["record_id"] = orders_df["record_id"].astype(str).str.strip()
orders_df["status"] = orders_df["status"].astype(str).str.strip()
orders_df["order_value_inr"] = pd.to_numeric(orders_df["order_value_inr"], errors="raise")
orders_df["days_since_created"] = pd.to_numeric(orders_df["days_since_created"], errors="raise")
orders_df["delayed_shipment"] = (
    orders_df["delayed_shipment"].astype(str).str.strip().str.lower()
    .map({"true": True, "false": False})
)
if orders_df["delayed_shipment"].isna().any():
    raise ValueError("delayed_shipment must contain only True/False.")

DELAY_WEIGHT = 0.70
RECENCY_WEIGHT = 0.30
MAX_ORDER_AGE_DAYS = 30
ESCALATION_THRESHOLD = float(
    (DELAY_WEIGHT * orders_df["delayed_shipment"].astype(float)
     + RECENCY_WEIGHT * (orders_df["days_since_created"] / MAX_ORDER_AGE_DAYS)).quantile(0.80)
)

def check_order_status(record_id: str) -> dict:
    if not record_id or not str(record_id).strip():
        return {"found": False, "error": "Please provide an order ID."}
    order_id = str(record_id).strip().upper()
    rows = orders_df[orders_df["record_id"].str.upper() == order_id]
    if rows.empty:
        return {"found": False, "record_id": order_id, "error": f"No order found for {order_id}."}
    row = rows.iloc[0]
    delayed = bool(row["delayed_shipment"])
    recency = float(row["days_since_created"]) / MAX_ORDER_AGE_DAYS
    score = round(float(np.clip(DELAY_WEIGHT * delayed + RECENCY_WEIGHT * recency, 0, 1)), 4)
    return {
        "found": True,
        "record_id": row["record_id"],
        "status": row["status"],
        "order_value_inr": float(row["order_value_inr"]),
        "escalation_score": score,
        "requires_escalation": score >= ESCALATION_THRESHOLD
    }

print("RAG chunks:", policy_collection.count())
print("Orders:", len(orders_df))
print("Escalation threshold:", round(ESCALATION_THRESHOLD, 4))


## 3. Create a JSON-backed conversation memory store

Each conversation has a unique ID. Messages are stored as a list of role/content records. The JSON file is read before a turn and updated after it, so memory survives a notebook restart.

The helper functions below deliberately keep conversations separate.

In [ ]:
def load_memory():
    if not MEMORY_PATH.exists():
        return {}
    try:
        data = json.loads(MEMORY_PATH.read_text(encoding="utf-8"))
        if not isinstance(data, dict):
            raise ValueError("Memory JSON root must be an object.")
        return data
    except json.JSONDecodeError as exc:
        raise ValueError(f"Memory file is not valid JSON: {MEMORY_PATH}") from exc


def save_memory(memory):
    MEMORY_PATH.write_text(
        json.dumps(memory, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )


def get_conversation_history(conversation_id):
    memory = load_memory()
    return memory.get(conversation_id, [])


def append_conversation_turn(conversation_id, user_message, assistant_message, metadata=None):
    memory = load_memory()
    history = memory.setdefault(conversation_id, [])
    timestamp = datetime.now(timezone.utc).isoformat()

    history.append({
        "role": "user",
        "content": user_message,
        "timestamp": timestamp
    })
    history.append({
        "role": "assistant",
        "content": assistant_message,
        "timestamp": timestamp,
        "metadata": metadata or {}
    })

    save_memory(memory)
    return history


print("Memory store initialized.")
print("Existing conversation IDs:", list(load_memory().keys()))


## 4. Define retrieval and order helpers

The RAG helper retrieves policy passages. The order helper extracts an order ID when present. In a follow-up turn, the agent can reuse the last order ID from that same conversation's stored history.

In [ ]:
ORDER_ID_PATTERN = re.compile(r"\bORD\d{4,}\b", flags=re.IGNORECASE)

ORDER_KEYWORDS = [
    "order status", "track my order", "track order",
    "where is my order", "check my order", "order update",
    "what is its status", "what's its status", "what is the status"
]

def retrieve_policy_context(query, top_k=TOP_K):
    query_embedding = embedding_model.encode(
        [query], normalize_embeddings=True
    )[0].tolist()

    result = policy_collection.query(
        query_embeddings=[query_embedding],
        n_results=min(top_k, policy_collection.count()),
        include=["documents", "metadatas", "distances"]
    )

    contexts = []
    for doc, meta, distance in zip(
        result.get("documents", [[]])[0],
        result.get("metadatas", [[]])[0],
        result.get("distances", [[]])[0]
    ):
        contexts.append({
            "text": doc,
            "source": meta.get("source", "unknown"),
            "similarity": round(1 - float(distance), 4)
        })
    return contexts


def find_last_order_id(history):
    # Search newest messages first; only use IDs explicitly present in this conversation.
    for message in reversed(history):
        match = ORDER_ID_PATTERN.search(message.get("content", ""))
        if match:
            return match.group(0).upper()
    return None


def classify_query(query):
    lower = query.lower()
    id_match = ORDER_ID_PATTERN.search(query)
    is_order_intent = bool(id_match) or any(term in lower for term in ORDER_KEYWORDS)
    return ("order" if is_order_intent else "policy"), (
        id_match.group(0).upper() if id_match else None
    )

print("Retrieval and order helpers ready.")


## 5. Build the memory-aware LangGraph

The graph uses four nodes:
- `load_conversation_memory`: loads the history for the active conversation ID.
- `classify_intent`: determines policy vs order intent and resolves a missing order ID from this conversation's history only.
- `run_tool`: calls the RAG retrieval function or order lookup function.
- `save_and_respond`: composes the response and persists the new turn.

The conversation ID is part of the graph state. A different ID gets a different history.

In [ ]:
class MemoryAgentState(TypedDict, total=False):
    conversation_id: str
    query: str
    history: list
    intent: Literal["policy", "order"]
    record_id: Optional[str]
    retrieved_context: list
    order_result: dict
    answer: str
    sources: list


def load_conversation_memory(state: MemoryAgentState) -> dict:
    conversation_id = state["conversation_id"]
    history = get_conversation_history(conversation_id)
    print(f"[load_memory] conversation={conversation_id}, messages={len(history)}")
    return {"history": history}


def classify_with_memory(state: MemoryAgentState) -> dict:
    intent, extracted_id = classify_query(state["query"])
    record_id = extracted_id

    if intent == "order" and record_id is None:
        record_id = find_last_order_id(state.get("history", []))

    print(f"[classify] intent={intent}, resolved_record_id={record_id}")
    return {"intent": intent, "record_id": record_id}


def run_selected_tool(state: MemoryAgentState) -> dict:
    if state["intent"] == "order":
        result = check_order_status(state.get("record_id"))
        return {"order_result": result, "retrieved_context": []}

    contexts = retrieve_policy_context(state["query"])
    return {"retrieved_context": contexts, "order_result": {}}


def save_and_respond(state: MemoryAgentState) -> dict:
    if state["intent"] == "order":
        result = state.get("order_result", {})
        if not result.get("found"):
            answer = result.get("error", "I could not find that order.")
            sources = []
        else:
            answer = (
                f"Order {result['record_id']} has status {result['status']}. "
                f"Order value: ₹{result['order_value_inr']:,.2f}. "
                f"Escalation score: {result['escalation_score']:.2f}. "
                + (
                    "This order is recommended for human review."
                    if result["requires_escalation"]
                    else "This order does not meet the project's escalation threshold."
                )
            )
            sources = ["orders dataset (synthetic project data)"]
    else:
        contexts = state.get("retrieved_context", [])
        if not contexts:
            answer = "I don't know based on the available policy documents."
            sources = []
        else:
            answer = (
                "According to the available project policy documents:\n\n"
                + "\n\n".join(
                    f"[{item['source']}] {item['text']}"
                    for item in contexts
                )
            )
            sources = list(dict.fromkeys(item["source"] for item in contexts))

    metadata = {
        "intent": state.get("intent"),
        "record_id": state.get("record_id"),
        "sources": sources
    }
    updated_history = append_conversation_turn(
        state["conversation_id"],
        state["query"],
        answer,
        metadata
    )
    print(f"[save_memory] persisted messages={len(updated_history)}")
    return {"answer": answer, "sources": sources, "history": updated_history}


memory_builder = StateGraph(MemoryAgentState)
memory_builder.add_node("load_conversation_memory", load_conversation_memory)
memory_builder.add_node("classify_intent", classify_with_memory)
memory_builder.add_node("run_tool", run_selected_tool)
memory_builder.add_node("save_and_respond", save_and_respond)

memory_builder.add_edge(START, "load_conversation_memory")
memory_builder.add_edge("load_conversation_memory", "classify_intent")
memory_builder.add_edge("classify_intent", "run_tool")
memory_builder.add_edge("run_tool", "save_and_respond")
memory_builder.add_edge("save_and_respond", END)

memory_agent = memory_builder.compile()
print("PASS: Four-node memory-aware LangGraph compiled.")


## 6. Demonstration A — Multi-turn conversation with memory

Turn 1 gives the agent an order ID. Turn 2 asks for the status without repeating the ID. The agent should recover that ID from the same conversation's JSON history and look up the order.

A dedicated demonstration conversation ID is used so this test can be rerun without mixing it with other conversations.

In [ ]:
CONTINUING_CONVERSATION_ID = "demo_customer_001"

# Reset only this demonstration conversation so the transcript is reproducible.
memory = load_memory()
memory.pop(CONTINUING_CONVERSATION_ID, None)
save_memory(memory)

turn1_query = "Please help me with order ORD1001."
turn1 = memory_agent.invoke({
    "conversation_id": CONTINUING_CONVERSATION_ID,
    "query": turn1_query
})

print("\nTURN 1 ANSWER:")
print(turn1["answer"])

turn2_query = "What is its status?"
turn2 = memory_agent.invoke({
    "conversation_id": CONTINUING_CONVERSATION_ID,
    "query": turn2_query
})

print("\nTURN 2 ANSWER:")
print(turn2["answer"])

assert turn2.get("record_id") == "ORD1001" or (
    turn2.get("order_result", {}).get("record_id") == "ORD1001"
)
assert turn2.get("order_result", {}).get("found") is True

print("\nPASS: Turn 2 reused ORD1001 from the same conversation's persisted history.")


## 7. Demonstration B — Fresh conversation has no previous state

Now use a new conversation ID and ask the same follow-up question without providing an order ID. The fresh conversation must not inherit the previous customer's order ID.

In [ ]:
FRESH_CONVERSATION_ID = "demo_customer_002"

# Ensure this is a new conversation with no stored history.
memory = load_memory()
memory.pop(FRESH_CONVERSATION_ID, None)
save_memory(memory)

fresh_query = "What is its status?"
fresh_result = memory_agent.invoke({
    "conversation_id": FRESH_CONVERSATION_ID,
    "query": fresh_query
})

print("\nFRESH CONVERSATION ANSWER:")
print(fresh_result["answer"])

assert fresh_result.get("order_result", {}).get("found") is False
assert "ORD1001" not in fresh_result["answer"]

print("\nPASS: Fresh conversation did not inherit the previous conversation's order ID.")


## 8. Inspect the persisted JSON memory

This cell prints the two separate conversation records and confirms that the first conversation has multiple turns while the fresh conversation has no inherited order ID.

In [ ]:
persisted_memory = load_memory()

print("CONTINUING CONVERSATION HISTORY")
print(json.dumps(
    persisted_memory.get(CONTINUING_CONVERSATION_ID, []),
    indent=2,
    ensure_ascii=False
))

print("\nFRESH CONVERSATION HISTORY")
print(json.dumps(
    persisted_memory.get(FRESH_CONVERSATION_ID, []),
    indent=2,
    ensure_ascii=False
))

assert len(persisted_memory[CONTINUING_CONVERSATION_ID]) >= 4
assert len(persisted_memory[FRESH_CONVERSATION_ID]) == 2

print("\nPASS: Conversation history persisted in JSON.")
print("PASS: Multi-turn state carried within one conversation.")
print("PASS: Fresh conversation state remained separate.")
print("Memory file:", MEMORY_PATH.resolve())
